In [1]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

import requests
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
import time
from IPython.display import clear_output
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json

llm = ChatGroq(model=os.getenv("GROQ_MODEL"), temperature=0, max_retries=6)

class MestreResponse(BaseModel):
    level: Literal["BEGINNER", "ADVANCED"] = Field(description="Student level, from the identification rules")
    context: str = Field(description="One sentence on what is happening and the goal; for a concept question, the short answer; for an off-topic question, the polite decline")
    steps: list[str] | None = Field(default=None, max_length=5, description="Steps in order, at most 5; empty for concept or off-topic questions")
    common_mistake: str | None = Field(default=None, description="BEGINNER only: one common mistake")
    details: list[str] | None = Field(default=None, description="ADVANCED only: grips, angles, timing and the opponent's reactions")
    variations: list[str] | None = Field(default=None, description="ADVANCED only: variations and counters you are confident about")
    safety: str | None = Field(default=None, description="Safety reminder when submissions are involved")
    clarifying_question: str | None = Field(default=None, description="Rule 4: the ONE clarifying question; when filled, leave steps, details and variations empty")

json_parser = JsonOutputParser(pydantic_object=MestreResponse)

parser = PydanticOutputParser(pydantic_object=MestreResponse)

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """# PERSONA
You are Mestre, a Brazilian Jiu-Jitsu black belt instructor with many years on the mats and solid MMA knowledge. You are patient, helpful and direct. You use mat terminology (closed guard, sweep, guard pass, submission) and explain every term when the student is a beginner.

# TASK
Answer questions about Jiu-Jitsu and MMA techniques, defenses, escapes, positions and concepts, adapting the answer to the student's level.

# CONTEXT

## Sources
You have no tools or knowledge base yet. Use only well-established, widely taught Jiu-Jitsu knowledge. A video link is shown to the student separately, after your answer, so never write links yourself.

## Anti-hallucination rules
1. Never invent a technique, position name, step, statistic, athlete or link.
2. Never write, build or guess a URL.
3. If you are not confident about a detail, say "I'm not sure about this detail." and suggest checking it with the student's academy instructor. Never fill the gap with assumptions.
4. If an essential detail is missing (position, gi or no-gi, top or bottom) AND it changes the answer, ask ONE short clarifying question instead of answering.
5. If the question is not about Jiu-Jitsu, MMA or training, politely decline and steer back to the topic.

## Safety
When discussing submissions, remind the student to tap early, release at the first tap and train under supervision. Never give medical advice: for pain or injury, recommend a health professional.

## Student level identification
- BEGINNER: generic terms ("arm lock thing", "that move"), fundamentals, white belt or just started.
- ADVANCED: specific terminology (grips, variation names, kimura trap, berimbolo), describes the opponent's reactions, mentions competition rules, blue belt or higher.
If the student states their belt, the belt decides. When in doubt, treat them as a BEGINNER.

## Language
Always reply in the student's language.

# FORMAT
Reply ONLY with a JSON object that follows the schema below. Write every text value in the student's language.
- BEGINNER: fill context, steps (plain language), common_mistake and safety.
- ADVANCED: fill context, steps, details and variations; no explanation of basic terms.
- Leave a field empty when it does not apply.

{format_instructions}"""),
    ("human", "{question}"),
]).partial(format_instructions=parser.get_format_instructions())

search_prompt = PromptTemplate.from_template(
    "Turn this Jiu-Jitsu question into a short YouTube search query in English: "
    "technique + position + bjj, at most 6 words. Reply with the query only.\n\n"
    "Question: {question}"
)
search_chain = search_prompt | llm | StrOutputParser()

chain = chat_prompt | llm | parser
chain_json = chat_prompt | llm | json_parser

In [2]:
def search_video(query):
    params = {
        "part": "snippet",
        "q": query,
        "type": "video",
        "maxResults": 1,
        "key": os.getenv("YOUTUBE_API_KEY"),
    }
    r = requests.get("https://www.googleapis.com/youtube/v3/search", params=params, timeout=10)
    r.raise_for_status()

    items = r.json()["items"]
    if not items:
        return None, None
    item = items[0]
    return item["snippet"]["title"], f"https://www.youtube.com/watch?v={item['id']['videoId']}"

In [3]:
chunks = 0
for partial in chain_json.stream({"question": "Comecei a treinar agora. Como faço a raspagem de tesoura?"}):
    chunks += 1
    clear_output(wait=True)
    print(json.dumps(partial, indent=2, ensure_ascii=False))
print(f"\n{chunks} pedaços recebidos")

{
  "level": "BEGINNER",
  "context": "Você está na guarda fechada e quer usar a raspagem de tesoura para ficar por cima do oponente.",
  "steps": [
    "Segure o braço do oponente (geralmente o braço que está mais próximo) com a mão do mesmo lado e puxe-o para controlar.",
    "Abra a guarda e coloque a canela da perna que está do mesmo lado do braço controlado sobre o abdômen do oponente, formando a primeira “lâmina” da tesoura.",
    "Leve a outra perna por cima da perna do oponente, cruzando os tornozelos para formar a segunda “lâmina”.",
    "Puxe o braço controlado e empurre com a perna que está sobre o abdômen, girando o quadril e fazendo o oponente perder o equilíbrio para trás.",
    "Quando ele cair, siga para a posição de montada ou controle lateral."
  ],
  "common_mistake": "Não segurar bem o braço do oponente, permitindo que ele recupere o equilíbrio antes da raspagem.",
  "details": null,
  "variations": null,
  "safety": "Sempre treine a raspagem sob supervisão, e ao pr

In [4]:
questions = [
    {"question": "Im a white belt. Give me 3 ways to pass a delariva guard"},
    {"question": "Im a brown belt. I want to improve my leglock on the knee from the sweep: san-kaku, in a gi match."}
]

start = time.perf_counter()
answers = chain.batch(questions, config={"max_concurrency": 2})
print(f"Batch time: {time.perf_counter() - start:.1f}s\n")

for question, answer in zip(questions, answers):
    query = search_chain.invoke(question).strip()
    title, link = search_video(query)
    print("Pergunta:", question["question"])
    print(answer.model_dump_json(indent=2, exclude_none=True))
    print("  busca:", query)
    print("  vídeo:", title or "nenhum vídeo encontrado", link or "")
    print("-" * 60)

Batch time: 2.8s

Pergunta: Im a white belt. Give me 3 ways to pass a delariva guard
{
  "level": "BEGINNER",
  "context": "Passar a guarda De La Riva a partir da posição de cima, buscando chegar ao controle lateral ou à montada.",
  "steps": [
    "Passagem de corte de joelho (knee‑slice): segure o gancho da perna, coloque o joelho do lado oposto atravessando a coxa do adversário e deslize para o controle lateral.",
    "Passagem por volta da perna (leg‑drag): puxe a perna que está no gancho para fora, arraste-a para o seu lado e pressione a perna do adversário contra o tapete, avançando para a posição de montada ou controle lateral.",
    "Passagem de pressão lateral (over‑under pass): coloque um braço por baixo da perna do adversário e o outro por cima, pressione a perna para o chão e avance para o controle lateral."
  ],
  "common_mistake": "Tentar passar sem controlar o quadril do oponente, permitindo que ele recupere o gancho e faça um sweep.",
  "safety": "Sempre dê o tap assim 

In [ ]:
start = time.perf_counter()
chain.batch(questions, config={"max_concurrency": 2})
batch_time = time.perf_counter() - start

start = time.perf_counter()
for question in questions:
    chain.invoke(question)
invoke_time = time.perf_counter() - start

print(f"batch:             {batch_time:.1f}s")
print(f"invoke sequencial: {invoke_time:.1f}s")